In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut


In [37]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_validate

def reg_results_holdout(clf, X_train, y_train, X_test, y_test):
    
    classifier = clf.fit(X_train, np.ravel(y_train))
    metrics = [mean_squared_error, mean_absolute_error, r2_score ]
    
    pred_train = classifier.predict(X_train)
    pred_test = classifier.predict(X_test)

    results = pd.DataFrame()
    
    for m in metrics:
        result_train = m(y_train, pred_train)
        result_test = m(y_test, pred_test)
        
        dict_res = {'Treino': ["{:.3f}".format(result_train)], 'Teste': ["{:.3f}".format(result_test)]}
        res_aux = pd.DataFrame(dict_res, index = [str(m.__name__)])
        
        results = pd.concat([results, res_aux], axis = 0)
    
    return results

def reg_results_cv(clf, X_train, y_train, kcv):
    
    classifier = clf.fit(X_train, np.ravel(y_train))
    metrics = ['explained_variance', 'neg_mean_absolute_error', 'neg_mean_squared_error', 'neg_median_absolute_error', 'r2' ]
    
    results = pd.DataFrame()
    for m in metrics:
        cv_results = cross_validate(clf, X_train, np.ravel(y_train), scoring = m, cv = kcv, return_train_score = True)
        dict_res = {'' + m + '_Treino': cv_results['train_score'], 
                    '' + m + '_Teste': cv_results['test_score']}
        res_aux = pd.DataFrame(dict_res)
        results = pd.concat([results, res_aux], axis = 1)
        
    results_mean = np.transpose(pd.DataFrame(results.mean(), columns=['media']))
    results = pd.concat([results, results_mean], axis = 0)
    return results


def reg_grid_search(clf, X_train, y_train, params, score, cv):    
    grid = GridSearchCV(clf, params, scoring = score, cv = cv, return_train_score=True)
    grid_fitted = grid.fit(X_train, np.ravel(y_train))
    print ("Best score: %.4f" % grid_fitted.best_score_)
    print ("Best parameters: %s" % grid_fitted.best_params_)
    return grid_fitted, grid_fitted.best_estimator_, grid_fitted.cv_results_

In [14]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=100, metric='minkowski', p=2)

In [9]:
df_regressao = pd.read_csv('Bases/regressao_1.csv')

In [18]:
def data_info(data):        
    info = pd.DataFrame()
    info['var'] = data.columns
    info['# missing'] = list(data.isnull().sum())
    info['% missing'] = info['# missing'] / data.shape[0]
    info['types'] = list(data.dtypes)
    info['unique values'] = list(len(data[var].unique()) for var in data.columns)
    
    return info

In [11]:
data_info(df_regressao)

,var,# missing,% missing,types,unique values
0,v_1,0,0.0,float64,2491
1,v_2,0,0.0,float64,2490
2,v_3,0,0.0,float64,2491
3,v_4,0,0.0,float64,2494
4,v_5,0,0.0,float64,2493
5,v_6,0,0.0,float64,2489
6,v_7,0,0.0,float64,2490
7,v_8,0,0.0,float64,2490
8,v_9,0,0.0,float64,2495
9,v_10,0,0.0,float64,2491


In [35]:
reg_results_cv(knn,df_regressao.drop(['target'],axis=1),df_regressao['target'],5)

,explained_variance_Treino,explained_variance_Teste,neg_mean_absolute_error_Treino,neg_mean_absolute_error_Teste,neg_mean_squared_error_Treino,neg_mean_squared_error_Teste,neg_median_absolute_error_Treino,neg_median_absolute_error_Teste,r2_Treino,r2_Teste
0,0.646770,0.635418,-58.765368,-59.877131,-5470.249127,-5558.675669,-48.310378,-50.757517,0.646733,0.634870
1,0.641838,0.635919,-59.034541,-60.135988,-5494.149983,-5789.453790,-49.130937,-49.734264,0.641838,0.629099
2,0.631265,0.637475,-60.053952,-59.912341,-5728.570928,-5455.066695,-49.922872,-50.555539,0.631264,0.636548
3,0.652798,0.636401,-58.387665,-58.488401,-5347.665309,-5674.831311,-49.068177,-47.215072,0.652693,0.635937
4,0.641567,0.637773,-58.997032,-60.213013,-5527.532660,-5645.845783,-49.237832,-49.688775,0.641264,0.636524
media,0.642848,0.636597,-59.047712,-59.725375,-5513.633602,-5624.774650,-49.134039,-49.590233,0.642758,0.634595


Questão 2

In [22]:
df_regressao_2 = pd.read_csv('Bases/regressao_2.csv')

In [21]:
from sklearn.linear_model import ElasticNet
#se l1_ratio = 0, a norma é a l2
elastic = ElasticNet(alpha=0.01, l1_ratio=0.5)

In [38]:
reg_results_cv(elastic,df_regressao_2.drop(['target'],axis=1),df_regressao_2['target'],10)

C:\Users\diego\Anaconda3\lib\site-packages\sklearn\metrics\scorer.py:100: DeprecationWarning: Scoring method mean_squared_error was renamed to neg_mean_squared_error in version 0.18 and will be removed in 0.20.
  sample_weight=sample_weight)
C:\Users\diego\Anaconda3\lib\site-packages\sklearn\metrics\scorer.py:100: DeprecationWarning: Scoring method mean_squared_error was renamed to neg_mean_squared_error in version 0.18 and will be removed in 0.20.
  sample_weight=sample_weight)
C:\Users\diego\Anaconda3\lib\site-packages\sklearn\metrics\scorer.py:100: DeprecationWarning: Scoring method mean_squared_error was renamed to neg_mean_squared_error in version 0.18 and will be removed in 0.20.
  sample_weight=sample_weight)
C:\Users\diego\Anaconda3\lib\site-packages\sklearn\metrics\scorer.py:100: DeprecationWarning: Scoring method mean_squared_error was renamed to neg_mean_squared_error in version 0.18 and will be removed in 0.20.
  sample_weight=sample_weight)
C:\Users\diego\Anaconda3\lib\sit

,explained_variance_Treino,explained_variance_Teste,neg_mean_absolute_error_Treino,neg_mean_absolute_error_Teste,mean_squared_error_Treino,mean_squared_error_Teste,neg_median_absolute_error_Treino,neg_median_absolute_error_Teste,r2_Treino,r2_Teste
0,0.567156,0.558918,-7.991492,-8.293198,-100.531334,-99.744224,-6.850269,-7.755213,0.567156,0.554084
1,0.559710,0.615790,-8.058759,-7.827752,-100.565198,-99.267178,-7.068610,-5.909520,0.559710,0.615356
2,0.578500,0.450446,-7.974769,-8.319540,-98.747351,-116.040644,-6.790802,-6.655864,0.578500,0.437809
3,0.559372,0.620373,-7.970014,-8.540452,-99.359400,-110.392721,-6.740541,-7.826297,0.559372,0.614838
4,0.555476,0.652990,-8.055112,-7.618475,-101.490085,-90.532259,-6.977140,-6.559505,0.555476,0.651689
5,0.571397,0.496746,-8.161978,-6.733127,-103.006240,-76.549946,-7.092353,-5.229256,0.571397,0.496578
6,0.560899,0.606546,-7.951123,-8.582076,-99.629139,-107.622392,-6.891065,-7.845094,0.560899,0.604887
7,0.575838,0.474959,-7.991537,-8.467192,-99.919985,-104.983382,-6.880026,-7.708464,0.575838,0.459574
8,0.573311,0.500830,-8.026679,-7.991036,-99.696336,-106.344362,-7.003492,-6.325024,0.573311,0.499458
9,0.570573,0.530391,-7.945252,-8.643231,-99.086929,-112.511524,-6.737341,-7.525872,0.570573,0.529721


Notebook Vargas

In [26]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import random
random.seed(42)

In [27]:
dados = pd.read_csv("Bases/regressao_2.csv")
y = dados.target
X = dados.drop('target', axis=1)

mse_treino = []
mse_teste = []

kf = KFold(n_splits=10, random_state=42, shuffle=False)

for train_index, test_index in kf.split(dados):
    X_train, X_test = X.loc[train_index], X.loc[test_index]
    y_train, y_test = y.loc[train_index], y.loc[test_index]
    en = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)
    en.fit(X_train, y_train)
    
    y_pred = en.predict(X_train)
    mse_treino.append(mean_squared_error(y_train, y_pred))

    y_pred = en.predict(X_test)
    mse_teste.append(mean_squared_error(y_test, y_pred))
     
print(round(np.mean(mse_treino),2))
print(round(np.mean(mse_teste),2))

100.2
102.4


In [28]:
mse_teste

[99.74422361770786,
 99.26717785800598,
 116.04064442480767,
 110.39272097203066,
 90.5322586644934,
 76.54994592521015,
 107.62239182329208,
 104.98338163912024,
 106.34436163481756,
 112.51152427372922]